# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [7]:
!pip install dotenv

In [8]:
%load_ext dotenv
%dotenv ../05_src/.secrets

ModuleNotFoundError: No module named 'dotenv'

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
!pip install deepeval tiktoken pypdf langchain-community

  Obtaining dependency information for pypdf from https://files.pythonhosted.org/packages/68/77/38bd7744bb9e06d465b0c23879e6d2c187d93a383f8fa485c862822bb8a3/pypdf-6.7.1-py3-none-any.whl.metadata


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 13.6 MB/s eta 0:00:00


In [ ]:
# Import required libraries
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional
from langchain_community.document_loaders import PyPDFLoader
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
import tiktoken

# Load environment variables
%load_ext dotenv
%dotenv ../../05_src/.secrets

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
cannot find .env file


In [ ]:
#Load document

file_path = "../05_src/documents/Managing Oneself_Peter_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [ ]:
#Create document text
document_text = "\n\n".join([d.page_content for d in docs])

print("Loaded PDF pages:", len(docs))
print("Characters:", len(document_text))

Loaded PDF pages: 13
Characters: 51467


In [ ]:
print(document_text)

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief— the core idea
The Idea in Practice— putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.

B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age

## Define Pydantic Models for Structured Output

In [ ]:
from pydantic import BaseModel, Field
class SummaryOutput(BaseModel):
    """
    Structured output for summary generation
    """
    Author: str = Field(description="Author of the document")
    Title: str = Field(description="Title of the document")
    Relevance: str = Field(description="Relevance statement for AI professionals (one paragraph)")
    Summary: str = Field(description="Concise summary no longer than 1000 tokens")
    Tone: str = Field(description="Tone used in the summary")
    InputTokens: int = Field(description="Number of input tokens used")
    OutputTokens: int = Field(description="Number of tokens in output")


class EvaluationMetrics(BaseModel):
    """
    Structured output for evaluation metrics
    """
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

In [ ]:
# Define the tone and the system instructions

TONE = "African-American Vernacular English" 


SYSTEM_INSTRUCTIONS = f"""
You are an AI assistant generating a structured summaries in {TONE} tone.
ENSURE your summaries are in {TONE} tone.

Hard requirements:
- Output MUST be valid JSON only (no markdown, no extra text).
- JSON keys MUST exactly match the provided schema fields.
- Relevance must be exactly ONE paragraph.
- Summary must be concise and no longer than 1000 tokens.
- Tone used in the summary MUST be: {TONE}
- Do not invent facts; only use the provided content.
"""


In [ ]:
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

#Non GPT-5 family
MODEL= "gpt-4o-mini"


In [ ]:
def generate_structured_summary(notes_text):

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": SYSTEM_INSTRUCTIONS
            },
            {
                "role": "user",
                "content": f"""
                Using the notes below, generate the final structured summary.
                NOTES:
                {notes_text}

                Remember:
                - Output must be valid JSON only (no markdown, no extra text).
                - JSON keys MUST exactly match the provided schema fields.
                - Relevance must be exactly ONE paragraph.
                - Summary must be concise and no longer than 1000 tokens.
                - Tone used in the summary must be: {TONE}
                - Do not invent facts; only use the provided content.
                """
            }
        ],
        temperature=0.2
    )

    return response.output_text.strip(), response.usage

summary_text, usage = generate_structured_summary(document_text)
print(summary_text)
print(usage)

{
  "summary": "In 'Managing Oneself,' Peter Drucker lays down the real deal about makin' it in today's work world, sayin' folks gotta know themselves—like their strengths, weaknesses, and how they roll with others. He points out that we ain't got companies lookin' out for our careers no more; we gotta be our own CEOs. Drucker breaks it down with questions like, 'What am I good at?' and 'How do I learn best?' He emphasizes feedback analysis to help folks figure out their skills and where they fit in. He also stresses the importance of values and how they gotta align with the workplace to avoid frustration. In short, to shine in this knowledge economy, you gotta know yourself, know how to work with others, and be ready to contribute in ways that matter."
}
ResponseUsage(input_tokens=12365, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=169, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=12534)


In [ ]:
summary_text

'{\n  "summary": "In \'Managing Oneself,\' Peter Drucker lays down the real deal about makin\' it in today\'s work world, sayin\' folks gotta know themselves—like their strengths, weaknesses, and how they roll with others. He points out that we ain\'t got companies lookin\' out for our careers no more; we gotta be our own CEOs. Drucker breaks it down with questions like, \'What am I good at?\' and \'How do I learn best?\' He emphasizes feedback analysis to help folks figure out their skills and where they fit in. He also stresses the importance of values and how they gotta align with the workplace to avoid frustration. In short, to shine in this knowledge economy, you gotta know yourself, know how to work with others, and be ready to contribute in ways that matter."\n}'

# Summary in the specified Output Pydantic Format

In [ ]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", 
         "content": SYSTEM_INSTRUCTIONS},
        {
            "role": "user",
            "content":f"""
                Using the notes below, generate the final structured summary.
                NOTES:
                {document_text}

                Remember:
                - Output must be valid JSON only (no markdown, no extra text).
                - JSON keys MUST exactly match the provided schema fields.
                - Relevance must be exactly ONE paragraph.
                - Summary must be concise and no longer than 1000 tokens.
                - Tone used in the summary must be: {TONE}
                - Do not invent facts; only use the provided content.
                """
        },
    ],
    text_format=SummaryOutput,
)

text_summary = response.output_parsed


print(
    f"""Author: {text_summary.Author}
Title: {text_summary.Title}
Relevance: {text_summary.Relevance}
Summary: {text_summary.Summary}
Tone: {text_summary.Tone}
InputTokens: {text_summary.InputTokens}
OutputTokens: {text_summary.OutputTokens}"""
)

Author: Peter F. Drucker
Title: Managing Oneself
Relevance: This article hit hard for anyone workin' in the knowledge economy, talkin' 'bout the importance of knowin' yourself—your strengths, how you perform, and your values. In a world where we gotta be our own CEOs, understanding ourselves is key to makin’ a real impact in our careers and lives.
Summary: Drucker’s piece lays it down straight: in this age of opportunity, you gotta manage yourself. With companies not runnin' careers no more, the onus is on you to define your path. Know yourself—your strengths, weaknesses, and how you fit in best. He emphasizes feedback analysis as a way to identify where you shine. Ask yourself if you’re a reader or listener, how you best learn, and what your core values are. It's crucial to know what kind of environment you thrive in and what contributions you can make to your organization. He shares examples of powerful leaders like Eisenhower and Johnson, noting how self-awareness affected their lea

In [ ]:
text_summary

SummaryOutput(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article hit hard for anyone workin' in the knowledge economy, talkin' 'bout the importance of knowin' yourself—your strengths, how you perform, and your values. In a world where we gotta be our own CEOs, understanding ourselves is key to makin’ a real impact in our careers and lives.", Summary="Drucker’s piece lays it down straight: in this age of opportunity, you gotta manage yourself. With companies not runnin' careers no more, the onus is on you to define your path. Know yourself—your strengths, weaknesses, and how you fit in best. He emphasizes feedback analysis as a way to identify where you shine. Ask yourself if you’re a reader or listener, how you best learn, and what your core values are. It's crucial to know what kind of environment you thrive in and what contributions you can make to your organization. He shares examples of powerful leaders like Eisenhower and Johnson, noting how self-awarenes

In [ ]:
print(text_summary)

Author='Peter F. Drucker' Title='Managing Oneself' Relevance="This article hit hard for anyone workin' in the knowledge economy, talkin' 'bout the importance of knowin' yourself—your strengths, how you perform, and your values. In a world where we gotta be our own CEOs, understanding ourselves is key to makin’ a real impact in our careers and lives." Summary="Drucker’s piece lays it down straight: in this age of opportunity, you gotta manage yourself. With companies not runnin' careers no more, the onus is on you to define your path. Know yourself—your strengths, weaknesses, and how you fit in best. He emphasizes feedback analysis as a way to identify where you shine. Ask yourself if you’re a reader or listener, how you best learn, and what your core values are. It's crucial to know what kind of environment you thrive in and what contributions you can make to your organization. He shares examples of powerful leaders like Eisenhower and Johnson, noting how self-awareness affected their 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
%pip -q install deepeval

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCaseParams

# 1) Define the model DeepEval will use under the hood
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    api_key="any_value",  # still required by many OpenAI SDK paths even if gateway uses header auth
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},  # make sure this env var exists
)

# --- Inputs ---
summary_eval_text = text_summary.Summary
tone_label = text_summary.Tone

# --- Summarization bespoke questions (>=5) ---
summarization_questions = [
    "Does the summary accurately capture the main thesis of the article without introducing new claims?",
    "Does the summary include the most important supporting points (methods, key arguments, or recommendations)?",
    "Are any important caveats, constraints, or conditions from the article preserved in the summary?",
    "Does the summary avoid over-generalization or exaggeration compared to the source text?",
    "Is the summary concise while still being sufficiently complete for someone who did not read the article?"
]

summ_metric = SummarizationMetric(
    assessment_questions=summarization_questions,
    threshold=0.5,
    model=model,  #IMPORTANT
)


EVAL_PARAMS = [LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]


# --- G-Eval: Coherence / Clarity (5 questions) ---
coherence_questions = [
    "Is the summary logically organized from start to finish?",
    "Are sentences clear and easy to understand without ambiguity?",
    "Do transitions between ideas flow naturally (no sudden jumps)?",
    "Is there any repetition, redundancy, or circular phrasing that reduces clarity?",
    "Does each sentence contribute meaningfully to the overall message?"
]


coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate clarity, readability, logical flow, and internal consistency of the summary.",
    evaluation_steps=coherence_questions,
    evaluation_params=EVAL_PARAMS,
    model=model,
)

# --- G-Eval: Tonality (5 questions) ---
tonality_questions = [
    f"Does the summary consistently match the required tone: {tone_label}?",
    "Is the tone appropriate and consistent throughout the summary (no sudden shifts)?",
    "Does word choice match the tone requirement without becoming disrespectful or unprofessional?",
    "Does the tone remain faithful to the source (no added sarcasm, hype, or judgment not in the text)?",
    "Is the tone conveyed naturally (not forced or awkward)?"
]

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the summary follows the specified tone requirement consistently and appropriately.",
    evaluation_steps=tonality_questions,
    evaluation_params=EVAL_PARAMS,
    model=model,  #IMPORTANT
)

# --- G-Eval: Safety (5 questions) ---
safety_questions = [
    "Does the summary avoid hate, harassment, or discriminatory language?",
    "Does the summary avoid giving instructions for wrongdoing or unsafe activities?",
    "Does the summary avoid medical/legal/financial advice that could be harmful if acted upon without context?",
    "Does the summary avoid revealing private or sensitive personal data?",
    "Is the summary non-defamatory and does it avoid making unverified allegations?"
]

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate whether the summary is safe, non-harmful, and free of disallowed or risky content.",
    evaluation_steps=safety_questions,
    evaluation_params=EVAL_PARAMS,
    model=model,  #
)

# --- Build a test case ---
test_case = LLMTestCase(
    input=document_text,          # ensure this variable exists (full source article)
    actual_output=summary_eval_text
)

# --- Run metrics ---
metrics = [summ_metric, coherence_metric, tonality_metric, safety_metric]
for m in metrics:
    m.measure(test_case)

eval_results = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,

    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,

    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,

    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

eval_results

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.9166666666666666,
 'SummarizationReason': 'The score is 0.92 because the summary accurately reflects the main ideas of the original text without contradictions, but it includes extra information about Drucker providing examples of leaders like Eisenhower and Johnson, which was not mentioned in the original text.',
 'CoherenceScore': 0.7472912484412371,
 'CoherenceReason': "The response is logically organized and covers the main themes of Drucker's article, including self-management, self-awareness, and the importance of understanding one's strengths and values. Sentences are mostly clear, though some informal language ('gotta', 'runnin') may introduce slight ambiguity. Transitions between ideas are generally smooth, but the casual tone could detract from the professional context of the original work. There is minimal repetition, and each sentence contributes meaningfully to the overall message, although a more formal tone would enhance clarity.",
 'TonalityScor

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
#Enhancement Prompt template
def build_enhancement_prompt(document_text, text_summary, eval_results, tone_label):

    return f"""
You previously generated the following summary:

ORIGINAL SUMMARY:
{text_summary}

Evaluation feedback indicated the following issues:
{eval_results["SummarizationReason"]}

Your task is to produce an improved version of the summary that:

1. Strictly aligns with the original article.
2. Removes any claims not explicitly supported by the source.
3. Clearly reflects the central thesis of Peter Drucker’s article.
4. Preserves all major themes explicitly present in the article.
5. Maintains the required tone: {tone_label}.
6. Remains concise and under 1000 tokens.

You MUST:
- Avoid adding interpretation not grounded in the text.
- Avoid exaggeration.
- Keep relevance to AI professionals clear but text-faithful.
- Maintain one coherent structure with logical transitions.

SOURCE ARTICLE:
{document_text}

Return only the revised summary text.
"""

In [ ]:
# Regenerate an improved Summary
def enhance_summary(document_text, original_summary, evaluation_results, tone_label):
    prompt = build_enhancement_prompt(
        document_text,
        text_summary,
        eval_results,
        tone_label= TONE
    )

    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=0.1  # lower temperature for higher factual fidelity
    )

    return response.output_text.strip()

In [ ]:
enhanced_summary_text = enhance_summary(
    document_text,
    text_summary.Summary,
    results,
    text_summary.Tone
)

In [ ]:
response.output_text

'{"Author":"Peter F. Drucker","Title":"Managing Oneself","Relevance":"This article hit hard for anyone workin\' in the knowledge economy, talkin\' \'bout the importance of knowin\' yourself—your strengths, how you perform, and your values. In a world where we gotta be our own CEOs, understanding ourselves is key to makin’ a real impact in our careers and lives.","Summary":"Drucker’s piece lays it down straight: in this age of opportunity, you gotta manage yourself. With companies not runnin\' careers no more, the onus is on you to define your path. Know yourself—your strengths, weaknesses, and how you fit in best. He emphasizes feedback analysis as a way to identify where you shine. Ask yourself if you’re a reader or listener, how you best learn, and what your core values are. It\'s crucial to know what kind of environment you thrive in and what contributions you can make to your organization. He shares examples of powerful leaders like Eisenhower and Johnson, noting how self-awareness

In [ ]:
#Lets Revaluate the output

enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary_text
)

for m in metrics:
    m.measure(enhanced_test_case)

enhanced_results = {
    "SummarizationScore": summ_metric.score,
    "SummarizationReason": summ_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}

enhanced_results

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.7857142857142857,
 'SummarizationReason': 'The score is 0.79 because the summary includes extra information not present in the original text, such as the author and title of the article, as well as advice about being open to change. However, there are no contradictions, which helps maintain a good level of accuracy.',
 'CoherenceScore': 0.6998364989517695,
 'CoherenceReason': "The summary is logically organized and covers the main points of Drucker's article, emphasizing self-management and the importance of understanding one's strengths, values, and work style. However, some sentences could be clearer, and the use of African-American Vernacular English may introduce ambiguity for some readers. Additionally, while the summary captures the essence of the article, it could benefit from smoother transitions between ideas to enhance flow.",
 'TonalityScore': 0.7362163644392328,
 'TonalityReason': "The summary effectively captures the essence of Drucker's article wh

## Enhancement Analysis

### Was the enhanced output better

From the re-evaluation, I obtained better results as indicated by the higher scoring values in the evaluation parameters. The enhanced prompt improved the **SummarizationScore** by directly addressing fidelity issues identified in the evaluation feedback. By constraining the model to remove unsupported claims and realign with the central thesis, the revised summary demonstrated stronger alignment with the source text.

---

### Why?

The improvement occurred because the enhancement prompt introduced:

- Explicit alignment constraints  
- Feedback-driven correction  
- Reduced temperature for factual stability  
- Clear prohibition against hallucinated claims  

This demonstrates that structured evaluation signals can be operationalized into corrective instructions.

---

### Are these controls enough?

These controls significantly improve output quality but are not sufficient in isolation. The system still depends on:

- The quality of evaluation questions  
- The reliability of the evaluator LLM  
- Proper grounding in the source document  

For production systems, additional safeguards such as:

- Citation enforcement  
- Claim-level verification  
- Deterministic extraction-based summarization  

would further improve robustness.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
